# Soft Sensor - CaO Livre (f-CaO)

Previsão da concentração de **CaO Livre** no clínquer a partir de variáveis de processo do forno

## Bibliotecas e imports

In [1]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [2]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

import warnings
warnings.filterwarnings('ignore')

DIR_DATA = os.getcwd() + "/data/"

## Funções utilitárias

Definições reutilizadas em todo o notebook: avaliação padronizada de métricas (`avaliar_modelo`), plotagem dos resultados (`plotar_resultados`), importância de variáveis e o registro central usado no comparativo final.

In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Registro central das métricas de cada experimento
resultados_experimentos = []


def plot_subplots(dados, vars_plot1, vars_plot2):
    """Plot interativo (Plotly) com dois painéis empilhados e eixo X compartilhado

    vars_plot1 vão no painel superior; vars_plot2 (ex.: o target) no inferior.
    """
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)
    for var in vars_plot1:
        fig.add_trace(go.Scatter(x=dados.index, y=dados[var], name=var, mode='lines'), row=1, col=1)
    for var in vars_plot2:
        fig.add_trace(go.Scatter(x=dados.index, y=dados[var], name=var, mode='lines', line_color='black'), row=2, col=1)
    fig.update_layout(
        height=700,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5, title=None),
    )
    return fig

# fig_subplot = plot_subplots(
#     dados=df_dataset,  
#     vars_plot1=['VAR1'], 
#     vars_plot2=['VAR2']
# )
# fig_subplot.show()


def calcular_metricas(y_true, y_pred):
    """Retorna um dicionário com MAE, RMSE e R² para um par (real, previsto)."""
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }


def avaliar_modelo(nome_modelo, y_train, preds_train, y_test, preds_test, registrar=True):
    """Avalia um experimento nas bases de treino e teste de forma padronizada

    Imprime R²/RMSE/MAE e, se ``registrar=True``, guarda o resultado em
    ``resultados_experimentos`` para o comparativo final. Reexecutar a célula
    sobrescreve o registro anterior de mesmo ``nome_modelo``
    """
    m_train = calcular_metricas(y_train, preds_train)
    m_test = calcular_metricas(y_test, preds_test)

    print(f"=== {nome_modelo} ===")
    print(f"[Treino] R2: {m_train['R2']:7.4f} | RMSE: {m_train['RMSE']:7.4f} | MAE: {m_train['MAE']:7.4f}")
    print(f"[Teste ] R2: {m_test['R2']:7.4f} | RMSE: {m_test['RMSE']:7.4f} | MAE: {m_test['MAE']:7.4f}")

    if registrar:
        resultados_experimentos[:] = [r for r in resultados_experimentos if r['Modelo'] != nome_modelo]
        resultados_experimentos.append({
            'Modelo': nome_modelo,
            'R2 Treino': m_train['R2'], 'RMSE Treino': m_train['RMSE'], 'MAE Treino': m_train['MAE'],
            'R2 Teste': m_test['R2'], 'RMSE Teste': m_test['RMSE'], 'MAE Teste': m_test['MAE'],
        })
    return m_train, m_test


def plotar_resultados(y_train, preds_train, y_test, preds_test, nome_modelo):
    """Plota Real vs Previsto na base de treino (topo) e de teste (base)"""
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharey=True)

    axes[0].plot(y_train.index, np.asarray(y_train).ravel(), label='Real (Laboratório)',
                 marker='o', linestyle='-', color='black', alpha=0.7)
    axes[0].plot(y_train.index, preds_train, label=f'Previsão {nome_modelo}',
                 marker='x', linestyle='--', color='blue', alpha=0.8)
    axes[0].set_title(f'{nome_modelo} - Base de Treino')
    axes[0].set_ylabel('CaO Livre')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.6)

    axes[1].plot(y_test.index, np.asarray(y_test).ravel(), label='Real (Laboratório)',
                 marker='o', linestyle='-', color='black', alpha=0.7)
    axes[1].plot(y_test.index, preds_test, label=f'Previsão {nome_modelo}',
                 marker='s', linestyle='-.', color='darkorange', alpha=0.8)
    axes[1].set_title(f'{nome_modelo} - Base de Teste')
    axes[1].set_ylabel('CaO Livre')
    axes[1].set_xlabel('Tempo')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()


def tabela_importancias(modelo, feature_names, top=15):
    """Retorna as ``top`` variáveis mais importantes de um modelo baseado em árvores"""
    return (
        pd.DataFrame({'Feature': list(feature_names),
                      'Importancia (%)': modelo.feature_importances_ * 100})
        .sort_values('Importancia (%)', ascending=False)
        .head(top)
        .reset_index(drop=True)
    )


def tabela_resultados():
    """Consolida os experimentos avaliados, ordenados pelo menor RMSE de teste"""
    if not resultados_experimentos:
        print("Nenhum experimento avaliado ainda. Rode as células de modelagem antes")
        return None
    return pd.DataFrame(resultados_experimentos).sort_values('RMSE Teste').reset_index(drop=True)

# Parâmetros

In [4]:
base_name = "CaO Liv.csv" # CaO Liv.csv || CaO Liv GERAL.csv
timestamp = "Timestamp"
process_name = "CaO Liv"
company_name = "CSN Cimentos"
subsistema = DIR_DATA + process_name +'_subsistema.csv'

target_col = 'AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS'

# Carregamento dos dados

Leitura do CSV, reamostragem para 1 minuto e indexação temporal.

In [ ]:
df_dataset = pd.read_csv(DIR_DATA + base_name, sep=";", decimal=".", encoding="utf-8-sig")

df_dataset[timestamp] = pd.to_datetime(df_dataset[timestamp], format="%Y-%m-%d %H:%M:%S")
df_dataset = df_dataset.set_index(timestamp)
df_dataset = df_dataset.resample('1min').asfreq()
df_dataset = df_dataset.reset_index()

list_columns_drop = []  # AR.F1.SI12501, AR.F1.CONS_CALOR e AR.F1.PI52606 agora possuem historico na base e sao mantidas como variaveis de processo
df_dataset.drop(list_columns_drop, axis=1, inplace=True, errors='ignore')

df_dataset.head()

In [ ]:
for col in df_dataset.columns:
    if col != 'Timestamp':
        df_dataset[col] = pd.to_numeric(df_dataset[col], errors='coerce')
df_dataset

## TAGs do subsistema CaO Livre

In [ ]:
df_sistema = pd.read_csv(subsistema, sep=";")
df_sistema

# Seleção do subconjunto

Recorte do período com amostragem consistente de 1 minuto.

In [ ]:
start_date = pd.to_datetime("2026-01-01 00:00:00")
end_date = pd.to_datetime("2026-07-15 10:20:00")
mask = (df_dataset[timestamp] >= start_date) & (df_dataset[timestamp] <= end_date)
df_dataset = df_dataset.loc[mask]
df_dataset.set_index(timestamp, inplace=True)
df_dataset

## Tratamento das variáveis de laboratório

Definição das variáveis de laboratório presentes na base de dados

In [ ]:
# list_lab_vars = ["AR.F1.MFARINHA_RET#170_LIMS","AR.F1.SAIDA_SILO_FARINHA_FSC_LIMS", "AR.F1.SAIDA_SILO_FARINHA_MS_LIMS", "AR.F1.SAIDA_SILO_FARINHA_MA_LIMS", "AR.F1.MFARINHA_FSC_LIMS", "AR.F1.CLINQUER_FL-1338_SRESF_LIMS", "AR.F1.CLINQUER_REL_SMF_SRESF_LIMS", "AR.F1.ULT_EST_FARINHA_GRAU_DESCARB_LIMS", "AR.F1.INDICE_VOL", "AR.F1.MCOQUE_PCI_LIMS", "AR.F1.MCOQUE_FIN#170_LIMS", "AR.F1.MCOQUE_S_LIMS", "AR.F1.MCOQUE_UMIDADE_LIMS", "AR.F1.MIXCOMB_CINZAS_MCOMB1_LIMS", "AR.F1.CLINQUER_EXCESSO_S_SRESF_LIMS", "AR.F1.CLINQUER_C3S_SRESF_LIMS", "AR.F1.CLINQUER_PESO_LIT_SRESF_LIMS", "AR.F1.CLINQUER_MS_SRESF_LIMS", "AR.F1.CLINQUER_MA_SRESF_LIMS", "AR.F1.CLINQUER_FE2O3_SRESF_LIMS", "AR.F1.CLINQUER_MGO_SRESF_LIMS", "AR.F1.CLINQUER_SIO2_SRESF_LIMS", "AR.F1.CLINQUER_FSC_SRESF_LIMS"]
# df_dataset.drop(list_lab_vars, axis=1, inplace=True, errors='ignore')
# df_dataset.head()

# Análise Exploratória (EDA)

## Informações básicas do dataset

In [ ]:
# Dimensões do Dataset
print(df_dataset.shape)

In [ ]:
# Informações das Colunas
df_dataset.info()

## Verificação de valores ausentes

In [ ]:
# Verificação de NaN
missing_data = df_dataset.isnull().sum()
missing_percent = (missing_data / len(df_dataset)) * 100

df_missing = pd.DataFrame({'Valores Ausentes': missing_data, '% Ausente': missing_percent})
df_missing.sort_values(by='% Ausente', ascending=False, inplace=True)
df_missing.head()

In [ ]:
# Removendo tags com muitos valores faltantes
df_dataset.drop([], axis=1, inplace=True, errors='ignore')
df_dataset.head()

## Variáveis de baixa variância

Identifica constantes e setpoints (baixo coeficiente de variação) candidatos a descarte.

In [ ]:
# Análise de Variância (Filtro de Constantes e Setpoints)

# Isola as variáveis preditoras
features = [col for col in df_dataset.columns if col not in [target_col]]

# Calcula o Desvio Padrão e a Média
df_var = pd.DataFrame({
    'Desvio Padrão': df_dataset[features].std(),
    'Média': df_dataset[features].mean()
})

# Calcula o Coeficiente de Variação (CV) em %: (Std / |Mean|) * 100
# Evita divisão por zero preenchendo com NaN onde a média for exatamente zero
df_var['CV (%)'] = (df_var['Desvio Padrão'] / df_var['Média'].replace(0, np.nan).abs()) * 100

# Ordena das variáveis MENOS variáveis para as mais variáveis
df_var = df_var.sort_values(by='CV (%)', ascending=True)

# Exibe as 20 variáveis mais estáveis (menor CV)
print("Top 20 variáveis mais estáveis (Candidatas a descarte):")
display(df_var.head(20).style.format("{:.4f}").background_gradient(cmap='Reds_r', subset=['CV (%)']))

In [ ]:
# Removendo tags com baixa variação
df_dataset.drop([], axis=1, inplace=True, errors='ignore') # Nenhuma TAG removida devido baixa variação
df_dataset.head()

In [ ]:
df_dataset.ffill(inplace=True)

## Retirada de desligados do forno

In [ ]:
print(f"Total de linhas antes do tratamento de parada: {len(df_dataset)}")

# Identifica os momentos em que o forno está parado (Corrente/Velocidade < 200)
mask_parada = df_dataset['AR.F1.II12501'] < 200

# Conta quantos minutos o forno ficou parado
minutos_parados = mask_parada.sum()
print(f"Minutos com o forno parado (< 200): {minutos_parados}")

# Substitui todos os dados de processo por NaN durante a parada
# Isso garante que qualquer cálculo de Lag ou Janela (rolling) que passe por esse período
# resulte em NaN e seja posteriormente descartado pelo dropna()
features_para_limpar = [col for col in df_dataset.columns if col not in [timestamp]]
df_dataset.loc[mask_parada, features_para_limpar] = np.nan

print("Valores de parada substituídos por NaN para quebrar o histórico de features.")

## Análise da variável target

In [ ]:
# Estatísticas básicas e distribuição da medida de laboratório
display(df_dataset.describe())

plt.figure(figsize=(10, 5))
sns.histplot(df_dataset[target_col].dropna(), kde=False, bins=30, color='blue')
plt.title('Distribuição da Variável Target (Medidas de Laboratório)')
plt.xlabel(target_col)
plt.ylabel('Frequência')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### Intervalo de amostragem do laboratório

In [ ]:
# 1. Isola a série do target, removendo temporariamente os NaNs (que incluem os períodos de parada)
alvos_validos = df_dataset[target_col].dropna()

# 2. Aplica a verificação de diferença apenas nos dados válidos
# Isso impede que os milhares de minutos de "NaN" sejam contabilizados como medições
mask_novas_medicoes = alvos_validos.diff() != 0
mask_novas_medicoes.iloc[0] = True # Garante que a primeira medição do dataset não seja descartada

# 3. Extrai os instantes de tempo exatos (index) apenas dessas novas medições reais
instantes_reais = alvos_validos[mask_novas_medicoes].index.to_series()

# 4. Calcula a diferença de tempo e converte para minutos
intervalos_minutos = instantes_reais.diff().dt.total_seconds() / 60
intervalos_minutos = intervalos_minutos.dropna() # Remove o primeiro NaN do diff de datas para a estatística

# Resultados
print("--- Estatísticas do Intervalo de Medição do Laboratório (em minutos) ---")
print(f"Média:   {intervalos_minutos.mean():.1f} minutos")
print(f"Mediana: {intervalos_minutos.median():.1f} minutos")
print(f"Mínimo:  {intervalos_minutos.min():.1f} minutos")
print(f"Máximo:  {intervalos_minutos.max():.1f} minutos")

# Visualização da Distribuição
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
intervalos_minutos.plot(kind='hist', bins=50, color='#1f77b4', edgecolor='black')
plt.title(f'Distribuição dos Intervalos de Medição - {target_col}')
plt.xlabel('Minutos entre Medições')
plt.ylabel('Frequência (Quantidade de vezes)')
plt.axvline(intervalos_minutos.median(), color='red', linestyle='--', label=f'Mediana ({intervalos_minutos.median():.0f} min)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Recuperação das medições reais

Desfaz o platô (forward-fill) para isolar os instantes reais de medição do laboratório.

In [ ]:
# Desfazendo o platô (forward-fill) para recuperar os instantes reais de medição

# Identifica as linhas onde o valor atual é diferente do valor da linha anterior.
# Isso marca o momento exato em que um novo resultado de laboratório entrou no sistema.
mudanca_valor = df_dataset[target_col].diff() != 0

# Garante que o primeiro valor não seja descartado
mudanca_valor.iloc[0] = True 

# Cria uma nova coluna de target "limpa" (apenas os instantes de medição, o resto vira NaN)
target_limpo = target_col + '_CLEAN'
df_dataset[target_limpo] = df_dataset[target_col].where(mudanca_valor, np.nan)

# Verifica o resultado
print(f"Total de linhas no dataset: {len(df_dataset)}")
print(f"Medições originais (com platô): {df_dataset[target_col].notnull().sum()}")
print(f"Medições reais resgatadas (sem platô): {df_dataset[target_limpo].notnull().sum()}")

df_plot = df_dataset[target_limpo].dropna()
plt.figure(figsize=(15, 4))
# O parâmetro linestyle='-' é o responsável por desenhar a linha conectando os pontos
plt.plot(df_plot.index, df_plot.values, marker='', linestyle='-', color='black', linewidth=1.5, label='Medição Real')

plt.title('Variável target apenas com medições reais')
plt.xlabel('Tempo')
plt.ylabel(target_col)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## Informação Mútua com lags

Varre lags de 0 a 180 min e mantém, por variável, o lag de maior informação mútua.

In [ ]:
# Tabela ordenada (Top Informação Mútua)
from sklearn.feature_selection import mutual_info_regression
import numpy as np
import pandas as pd

max_lag = 180

# Isola as variáveis preditoras
features = [col for col in df_dataset.columns if col not in [target_col, target_limpo]]

# 1. Trata NaNs nas features contínuas para o cálculo não quebrar
df_features_filled = df_dataset[features].ffill().bfill()

# --- DEFINIÇÃO DO CORTE TREINO/TESTE (evita vazamento na seleção de lags) ---
# A seleção de lags por Informação Mútua NÃO pode enxergar o período de teste.
# Definimos aqui a data de corte (80% das medições de laboratório) e a reutilizamos
# no split do modelo, garantindo coerência e ausência de look-ahead.
_instantes_lab = df_dataset[target_limpo].dropna().index
_cut = max(1, int(len(_instantes_lab) * 0.8))
train_cutoff_date = _instantes_lab[_cut - 1]
print(f'Data de corte treino/teste: {train_cutoff_date} '
      f'({_cut} medições de treino / {len(_instantes_lab) - _cut} de teste)')

# Isola apenas os instantes de tempo onde existe laudo real do laboratório
y_target = df_dataset[target_limpo]
indices_validos = y_target.dropna().index
indices_validos = indices_validos[indices_validos <= train_cutoff_date]  # SOMENTE TREINO
y_valid = y_target.loc[indices_validos]

# Preparação da matriz
mi_matrix = np.zeros((len(features), max_lag + 1))

print("Calculando Informação Mútua por lag (isso pode levar alguns minutos)...")

# Cálculo da Informação Mútua por lag
for lag in range(max_lag + 1):
    # Desloca as features no tempo
    shifted_features = df_features_filled.shift(lag)
    
    # Filtra as features deslocadas para coincidir com os instantes do laudo
    X_valid = shifted_features.loc[indices_validos]
    
    # O shift pode gerar NaNs nas primeiras linhas; tratamos com backward fill
    X_valid = X_valid.bfill().fillna(0) 
    
    # Calcula a MI para todas as features de uma vez contra o target contínuo
    mi_scores = mutual_info_regression(X_valid, y_valid, random_state=42)
    mi_matrix[:, lag] = mi_scores

# Converte para DataFrame
df_lags_mi = pd.DataFrame(mi_matrix, index=features, columns=range(max_lag + 1))

# Transformação e ordenação para a tabela
df_tabela_mi = df_lags_mi.unstack().reset_index()
df_tabela_mi.columns = ['Lag (min)', 'Variável', 'Informação Mútua']
df_tabela_mi = df_tabela_mi.sort_values(by='Informação Mútua', ascending=False)

# Mantém apenas a primeira ocorrência (o lag de maior MI) de cada variável
df_tabela_mi = df_tabela_mi.drop_duplicates(subset=['Variável'], keep='first').reset_index(drop=True)

# Exibição do resultado
display(df_tabela_mi.head(20).style.background_gradient(cmap='viridis', subset=['Informação Mútua']))

# Experimento 1 - Features por lags de correlação

Cada variável de processo entra com o lag de maior informação mútua encontrada, somada à última medição de laboratório disponível.

## Construção do dataset com lags

In [ ]:
print("Construindo matriz de features alinhadas no tempo com lags ótimos de informação mútua...")

# Cria um novo DataFrame vazio mantendo o index original (minuto a minuto)
X_features = pd.DataFrame(index=df_dataset.index)

# 1. --- VARIÁVEIS DE PROCESSO CONTÍNUO ---
# Itera sobre a tabela de maiores correlações para aplicar o respectivo lag
for _, row in df_tabela_mi.iterrows():
    var_nome = row['Variável']
    lag_otimo = int(row['Lag (min)'])
    col_name = f"{var_nome}_lag{lag_otimo}"
    X_features[col_name] = df_dataset[var_nome].shift(lag_otimo)

# 2. --- ÚLTIMA MEDIÇÃO DO TARGET ---
# Propaga a última medição válida (ffill) e desloca 1 minuto para o futuro (shift)
X_features['AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS_lag1'] = df_dataset[target_limpo].ffill().shift(1)

# 4. Adiciona a variável target limpa ao novo dataset
dataset_modelagem = X_features.join(df_dataset[target_limpo])

print(f"Dataset construído com {X_features.shape[1]} features.")
dataset_modelagem.head()

## Split temporal treino/teste

In [ ]:
# Isolamento das medições de laboratório e Train/Test Split

# Filtra APENAS os instantes onde temos a resposta do laboratório
df_modelos = dataset_modelagem.dropna(subset=[target_limpo]).copy()

# Remove as primeiras linhas que ficaram com NaN devido ao .shift() dos lags
df_modelos = df_modelos.dropna()

print(f"Total de amostras válidas para treinamento e validação: {len(df_modelos)}")

# Divisão Temporal — usa a MESMA data de corte da seleção de lags (sem vazamento)
train_set = df_modelos[df_modelos.index <= train_cutoff_date]
test_set = df_modelos[df_modelos.index > train_cutoff_date]

X_train = train_set.drop(columns=[target_limpo])
y_train = train_set[target_limpo]

X_test = test_set.drop(columns=[target_limpo])
y_test = test_set[target_limpo]

print(f"Amostras de Treino: {len(X_train)} | Amostras de Teste: {len(X_test)}")

# Plot da divisão para confirmação visual
plt.figure(figsize=(15, 4))
plt.plot(y_train.index, y_train, label='Treino', color='blue', marker='o', linestyle='', markersize=4)
plt.plot(y_test.index, y_test, label='Teste', color='orange', marker='o', linestyle='', markersize=4)
plt.title('Divisão Temporal: Treino vs Teste')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## Padronização das features

O `StandardScaler` é ajustado apenas no treino para evitar vazamento de dados.

In [ ]:
# Padronização das features
from sklearn.preprocessing import StandardScaler

# O scaler é ajustado APENAS no treino para evitar vazamento de dados
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)

# O teste é transformado usando as regras aprendidas no treino
X_test_scaled = scaler_X.transform(X_test)

## Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

# Treina a regressão Ridge e gera previsões em treino e teste
modelo_ridge = Ridge(alpha=70.0)
modelo_ridge.fit(X_train_scaled, y_train)

preds_treino_ridge = modelo_ridge.predict(X_train_scaled)
preds_teste_ridge = modelo_ridge.predict(X_test_scaled)

avaliar_modelo("Ridge (lags)", y_train, preds_treino_ridge, y_test, preds_teste_ridge)

In [ ]:
plotar_resultados(y_train, preds_treino_ridge, y_test, preds_teste_ridge, "Ridge (lags)")

## PLS (Partial Least Squares)

In [ ]:
from sklearn.cross_decomposition import PLSRegression

modelo_pls = PLSRegression(n_components=4)
modelo_pls.fit(X_train_scaled, y_train)

# flatten() porque o PLS retorna um array 2D
preds_treino_pls = modelo_pls.predict(X_train_scaled).flatten()
preds_teste_pls = modelo_pls.predict(X_test_scaled).flatten()

avaliar_modelo("PLS (lags)", y_train, preds_treino_pls, y_test, preds_teste_pls)

In [ ]:
plotar_resultados(y_train, preds_treino_pls, y_test, preds_teste_pls, "PLS (lags)")

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# max_depth e min_samples_leaf limitam a memorização de ruído
modelo_rf = RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=4, random_state=42)
modelo_rf.fit(X_train_scaled, y_train)

preds_treino_rf = modelo_rf.predict(X_train_scaled)
preds_teste_rf = modelo_rf.predict(X_test_scaled)

avaliar_modelo("Random Forest (lags)", y_train, preds_treino_rf, y_test, preds_teste_rf)

In [ ]:
plotar_resultados(y_train, preds_treino_rf, y_test, preds_teste_rf, "Random Forest (lags)")

### Importância das variáveis

In [ ]:
# 15 variáveis mais determinantes para o Random Forest
df_importancia = tabela_importancias(modelo_rf, X_train.columns, top=15)
print("As 15 variáveis mais determinantes para o Soft Sensor:")
display(df_importancia.style.background_gradient(cmap='Greens', subset=['Importancia (%)']))

## Gradient Boosting

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

# l2_regularization penaliza variações bruscas, combatendo o overfitting
modelo_hgb = HistGradientBoostingRegressor(max_iter=150, max_depth=8, l2_regularization=0.2, random_state=42)
modelo_hgb.fit(X_train_scaled, y_train)

preds_treino_hgb = modelo_hgb.predict(X_train_scaled)
preds_teste_hgb = modelo_hgb.predict(X_test_scaled)

avaliar_modelo("Gradient Boosting (lags)", y_train, preds_treino_hgb, y_test, preds_teste_hgb)

In [ ]:
plotar_resultados(y_train, preds_treino_hgb, y_test, preds_teste_hgb, "Gradient Boosting (lags)")

## XGBoost

In [ ]:
import xgboost as xgb

# Parâmetros restritivos (profundidade baixa + L2 forte) para evitar memorizar ruído
modelo_xgb = xgb.XGBRegressor(
    n_estimators=150, max_depth=4, learning_rate=0.05, reg_lambda=5.0, random_state=42,
)
modelo_xgb.fit(X_train_scaled, y_train)

preds_treino_xgb = modelo_xgb.predict(X_train_scaled)
preds_teste_xgb = modelo_xgb.predict(X_test_scaled)

avaliar_modelo("XGBoost (lags)", y_train, preds_treino_xgb, y_test, preds_teste_xgb)

In [ ]:
plotar_resultados(y_train, preds_treino_xgb, y_test, preds_teste_xgb, "XGBoost (lags)")

## Regressão Linear Múltipla (MLR)

In [ ]:
# Treinamento da Regressão Linear Múltipla (MLR)
from sklearn.linear_model import LinearRegression

print("Treinando modelo de Regressão Linear Múltipla...")

modelo_mlr = LinearRegression()
modelo_mlr.fit(X_train_scaled, y_train)

# Previsões
preds_treino_mlr = modelo_mlr.predict(X_train_scaled)
preds_teste_mlr = modelo_mlr.predict(X_test_scaled)

# Avaliação padronizada
avaliar_modelo("MLR (Lags)", y_train, preds_treino_mlr, y_test, preds_teste_mlr)

In [ ]:
# Gráficos de Real vs Previsto
plotar_resultados(y_train, preds_treino_mlr, y_test, preds_teste_mlr, "Regressão Linear Múltipla (Lags)")

## Verificação de Concept Drift

Compara as distribuições do target entre treino e teste para detectar mudança de regime.

In [ ]:
# Análise de mudança de regime (concept drift) entre treino e teste
import seaborn as sns

# 1. Estatísticas descritivas comparativas
print("--- Comparativo Estatístico do Target (Laboratório) ---")
df_comparativo = pd.DataFrame({
    'Base de Treino': y_train.describe(),
    'Base de Teste': y_test.describe()
})
display(df_comparativo.style.format("{:.4f}"))

# 2. Visualização das distribuições
plt.figure(figsize=(12, 5))
sns.kdeplot(y_train, label='Distribuição Treino', fill=True, color='blue', alpha=0.4)
sns.kdeplot(y_test, label='Distribuição Teste', fill=True, color='orange', alpha=0.4)

# Linhas de média
plt.axvline(y_train.mean(), color='darkblue', linestyle='--', label=f'Média Treino: {y_train.mean():.2f}')
plt.axvline(y_test.mean(), color='darkorange', linestyle='--', label=f'Média Teste: {y_test.mean():.2f}')

plt.title('Diagnóstico de Concept Drift no Target')
plt.xlabel('Valores do Target')
plt.ylabel('Densidade')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Experimento 2 - Features por janelas estatísticas

Em vez de lags pontuais, resume cada variável por estatísticas móveis (média, desvio, mínimo, máximo) em janelas de 180 minutos.

## Feature Engineering (janelas de 180 min)

In [ ]:
# Extração de Estatísticas em Múltiplas Janelas Temporais
import pandas as pd

print("Calculando features para janelas...")

janelas = [180]
features_processo = [col for col in df_dataset.columns if col not in [target_col, target_limpo]]

lista_dfs_janelas = []

# Gera o rolling para cada janela e anexa à lista
for j in janelas:
    df_temp = df_dataset[features_processo].rolling(window=j).agg(['mean', 'std', 'min', 'max'])
    # Adiciona o indicador da janela no nome da coluna (ex: sensorX_mean_180m)
    df_temp.columns = [f"{col[0]}_{col[1]}_{j}m" for col in df_temp.columns]
    lista_dfs_janelas.append(df_temp)

# Concatena todas as 4 bases horizontalmente
df_multi_janelas = pd.concat(lista_dfs_janelas, axis=1)

# Propaga a última medição válida (ffill) e desloca 1 minuto para o futuro (shift)
df_multi_janelas['AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS_lag1'] = df_dataset[target_limpo].ffill().shift(1)

# Insere a variável target limpa e filtra APENAS as linhas de laudo
df_multi_janelas[target_limpo] = df_dataset[target_limpo]
df_modelos_multi = df_multi_janelas.dropna(subset=[target_limpo]).copy()

# Remove as amostras iniciais baseadas na MAIOR janela (180 min)
df_modelos_multi = df_modelos_multi.dropna()

print(f"Base multi-janelas criada: {df_modelos_multi.shape[0]} amostras com {df_modelos_multi.shape[1]-1} features.")

## Split temporal treino/teste

In [ ]:
# Divisão Cronológica (Train/Test Split)
split_index = int(len(df_modelos_multi) * 0.8)

train_set = df_modelos_multi.iloc[:split_index]
test_set = df_modelos_multi.iloc[split_index:]

X_train_multi = train_set.drop(columns=[target_limpo])
y_train_win = train_set[target_limpo]

X_test_multi = test_set.drop(columns=[target_limpo])
y_test_win = test_set[target_limpo]

print(f"Divisão -> Treino: {X_train_multi.shape[0]} amostras | Teste: {X_test_multi.shape[0]} amostras")

## Feature Selection

In [ ]:
# Feature Selection com Random Forest
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor

print("Executando Feature Selection...")

# Instancia um Random Forest como "avaliador" de importância
avaliador_rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)

# O seletor manterá as features cuja importância for maior que a mediana (threshold)
# Você pode restringir mais usando threshold='1.25*median' ou threshold='mean'
seletor = SelectFromModel(avaliador_rf, threshold='median')
seletor.fit(X_train_multi, y_train_win)

# Captura os nomes das features aprovadas
features_aprovadas = X_train_multi.columns[seletor.get_support()]

print(f"Total de features ORIGINAIS: {X_train_multi.shape[1]}")
print(f"Total de features MANTIDAS: {len(features_aprovadas)}")

# Filtra os DataFrames mantendo apenas as features que passaram no corte
X_train_sel = X_train_multi[features_aprovadas]
X_test_sel = X_test_multi[features_aprovadas]

# Exibe as Top 15 para entendermos quais tempos de janela o processo mais "escuta"
df_importancias = pd.DataFrame({
    'Feature': X_train_multi.columns, 
    'Importância': seletor.estimator_.feature_importances_
}).sort_values(by='Importância', ascending=False)

display(df_importancias.head(15))

## Padronização das features

In [ ]:
# Célula 4: Padronização da Base Selecionada (Scaling)
from sklearn.preprocessing import StandardScaler

scaler_sel = StandardScaler()

X_train_scaled = scaler_sel.fit_transform(X_train_sel)
X_test_scaled = scaler_sel.transform(X_test_sel)

# Reconstrução para DataFrame mantendo index e colunas
X_train_win_scaled = pd.DataFrame(X_train_scaled, index=X_train_sel.index, columns=X_train_sel.columns)
X_test_win_scaled = pd.DataFrame(X_test_scaled, index=X_test_sel.index, columns=X_test_sel.columns)

print("Matrizes prontas para o treinamento dos modelos!")

## Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

modelo_ridge = Ridge(alpha=50)
modelo_ridge.fit(X_train_win_scaled, y_train_win)

preds_treino_ridge = modelo_ridge.predict(X_train_win_scaled)
preds_teste_ridge = modelo_ridge.predict(X_test_win_scaled)

avaliar_modelo("Ridge (janelas)", y_train_win, preds_treino_ridge, y_test_win, preds_teste_ridge)

In [ ]:
plotar_resultados(y_train_win, preds_treino_ridge, y_test_win, preds_teste_ridge, "Ridge (janelas)")

## PLS (Partial Least Squares)

In [ ]:
from sklearn.cross_decomposition import PLSRegression

modelo_pls = PLSRegression(n_components=5)
modelo_pls.fit(X_train_win_scaled, y_train_win)

preds_treino_pls = modelo_pls.predict(X_train_win_scaled).flatten()
preds_teste_pls = modelo_pls.predict(X_test_win_scaled).flatten()

avaliar_modelo("PLS (janelas)", y_train_win, preds_treino_pls, y_test_win, preds_teste_pls)

In [ ]:
plotar_resultados(y_train_win, preds_treino_pls, y_test_win, preds_teste_pls, "PLS (janelas)")

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

modelo_rf = RandomForestRegressor(n_estimators=150, max_depth=5, min_samples_leaf=10,
                                  max_features='sqrt', random_state=42)
modelo_rf.fit(X_train_win_scaled, y_train_win)

preds_treino_rf = modelo_rf.predict(X_train_win_scaled)
preds_teste_rf = modelo_rf.predict(X_test_win_scaled)

avaliar_modelo("Random Forest (janelas)", y_train_win, preds_treino_rf, y_test_win, preds_teste_rf)

In [ ]:
plotar_resultados(y_train_win, preds_treino_rf, y_test_win, preds_teste_rf, "Random Forest (janelas)")

## XGBoost

In [ ]:
import xgboost as xgb

modelo_xgb = xgb.XGBRegressor(n_estimators=150, max_depth=4, learning_rate=0.05,
                              reg_lambda=10.0, random_state=42)
modelo_xgb.fit(X_train_win_scaled, y_train_win)

preds_treino_xgb = modelo_xgb.predict(X_train_win_scaled)
preds_teste_xgb = modelo_xgb.predict(X_test_win_scaled)

avaliar_modelo("XGBoost (janelas)", y_train_win, preds_treino_xgb, y_test_win, preds_teste_xgb)

In [ ]:
plotar_resultados(y_train_win, preds_treino_xgb, y_test_win, preds_teste_xgb, "XGBoost (janelas)")

# Experimento 3 - Deep Learning

Redes neurais LSTM sobre as sequências temporais brutas.

## LSTM com Attention

### Pré-processamento e sequências 3D

In [ ]:
# Construção das sequências 3D brutas (Tratadas apenas contra NaNs)
import numpy as np
import pandas as pd

print("Construindo sequências 3D...")

seq_length = 180

# 1. Isola os instantes dos laudos
df_alvos = df_dataset[[target_limpo]].dropna()

# 2. Isola as features e trata NaNs
features_processo = [col for col in df_dataset.columns if col not in [target_col, target_limpo]]
df_features_raw = df_dataset[features_processo].ffill().bfill()

# --- ADIÇÃO: Inclui a última medição de laboratório como feature contínua ---
# O ffill propaga o valor e o shift(1) garante que a rede não enxergue o laudo do tempo atual
df_features_raw['ultima_medicao_laboratorio'] = df_dataset[target_limpo].ffill().shift(1)
# Trata o NaN gerado na primeira linha pelo shift(1)
df_features_raw['ultima_medicao_laboratorio'] = df_features_raw['ultima_medicao_laboratorio'].bfill()

# 3. Construção dos arrays 3D
X_seq, y_seq, indices_validos = [], [], []

for tempo_alvo, row in df_alvos.iterrows():
    fim_idx = df_features_raw.index.get_loc(tempo_alvo)
    inicio_idx = fim_idx - seq_length + 1
    
    if inicio_idx >= 0:
        janela_3d = df_features_raw.iloc[inicio_idx : fim_idx + 1].values
        X_seq.append(janela_3d)
        y_seq.append(row[target_limpo])
        indices_validos.append(tempo_alvo)

X_seq = np.array(X_seq)
y_seq = np.array(y_seq).reshape(-1, 1)

# 4. Divisão Temporal (80% Treino / 20% Teste)
split_seq = int(len(X_seq) * 0.8)

X_train_seq, X_test_seq = X_seq[:split_seq], X_seq[split_seq:]
y_train_seq, y_test_seq = y_seq[:split_seq], y_seq[split_seq:]
idx_train, idx_test = indices_validos[:split_seq], indices_validos[split_seq:]

print(f"Shape X Treino: {X_train_seq.shape} | Shape y Treino: {y_train_seq.shape}")
print(f"Shape X Teste:  {X_test_seq.shape} | Shape y Teste:  {y_test_seq.shape}")

### DataLoaders

In [ ]:
# Criação dos DataLoaders do PyTorch
import torch
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_seq, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_seq, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

print("DataLoaders prontos.")

### Arquitetura Com Self-Attention

In [ ]:
# Definição da nova arquitetura da rede neural
import torch.nn as nn
import torch

class AdvancedSoftSensorLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super(AdvancedSoftSensorLSTM, self).__init__()
        
        # Solução 1: Instance Normalization (Normaliza cada sensor dentro da sua própria janela)
        # affine=False garante que não adicionamos parâmetros lineares fixos que possam enviesar regimes diferentes
        self.instance_norm = nn.InstanceNorm1d(input_dim, affine=False)
        
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        
        # Solução 3: Mecanismo de Atenção (Gera um score de importância para cada um dos 180 passos temporais)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        # x shape inicial: [batch, seq_len, features]
        
        # 1. Executa a Normalização por Instância
        # O módulo exige a dimensão de features na segunda posição: [batch, features, seq_len]
        x = x.permute(0, 2, 1)
        x = self.instance_norm(x)
        x = x.permute(0, 2, 1) # Retorna para: [batch, seq_len, features]
        
        # 2. Passa pela estrutura recorrente da LSTM
        lstm_out, _ = self.lstm(x) # lstm_out shape: [batch, seq_len, hidden_dim]
        
        # 3. Aplica o cálculo do Mecanismo de Atenção
        attn_scores = self.attention(lstm_out) # shape: [batch, seq_len, 1]
        attn_weights = torch.softmax(attn_scores, dim=1) # Normaliza os pesos no tempo
        
        # Multiplica os pesos pelos estados ocultos correspondentes e soma todo o histórico (Context Vector)
        context = torch.sum(attn_weights * lstm_out, dim=1) # shape: [batch, hidden_dim]
        
        # 4. Camadas de saída densas
        out = self.fc1(context)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

input_dim = X_train_tensor.shape[2]
modelo_avancado = AdvancedSoftSensorLSTM(input_dim)
print(modelo_avancado)

### Treinamento

In [ ]:
# Loop de treinamento com tratamento de estabilidade
import torch.optim as optim

optimizer = optim.AdamW(modelo_avancado.parameters(), lr=0.0005, weight_decay=0.01)
criterion = nn.MSELoss()

epochs = 100
train_losses, test_losses = [], []

print("Treinando LSTM com InstanceNorm e Atenção...")

for epoch in range(epochs):
    modelo_avancado.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = modelo_avancado(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        # Clipping para evitar explosão de gradiente com a inserção da atenção
        torch.nn.utils.clip_grad_norm_(modelo_avancado.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    
    modelo_avancado.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = modelo_avancado(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    epoch_val_loss = val_loss / len(test_loader.dataset)
    test_losses.append(epoch_val_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Época {epoch+1:03d}/{epochs} | Loss Treino: {epoch_loss:.4f} | Loss Teste: {epoch_val_loss:.4f}")

### Avaliação

In [ ]:
# Previsões da LSTM avançada e avaliação
modelo_avancado.eval()
with torch.no_grad():
    preds_treino = modelo_avancado(X_train_tensor).numpy().flatten()
    preds_teste = modelo_avancado(X_test_tensor).numpy().flatten()

y_train_series = pd.Series(y_train_seq.flatten(), index=idx_train)
y_test_series = pd.Series(y_test_seq.flatten(), index=idx_test)

avaliar_modelo("LSTM", y_train_series, preds_treino, y_test_series, preds_teste)
plotar_resultados(y_train_series, preds_treino, y_test_series, preds_teste, "LSTM")

## LSTM com arquitetura rasa

### Pré-processamento e sequências 3D

In [ ]:
# Construção das sequências 3D brutas (Tratadas apenas contra NaNs)
import numpy as np
import pandas as pd

print("Construindo sequências 3D...")

seq_length = 180

# 1. Isola os instantes dos laudos
df_alvos = df_dataset[[target_limpo]].dropna()

# 2. Separa as variáveis contínuas das variáveis esparsas de laboratório
features_processo = [col for col in df_dataset.columns if col not in [target_col, target_limpo]]

# Trata NaNs das variáveis de processo contínuo
df_features_raw = df_dataset[features_processo].ffill().bfill()


# --- ADIÇÃO 2: Última medição de CaO Livre ---
df_features_raw['ultima_medicao_laboratorio'] = df_dataset[target_limpo].ffill().shift(1)

# Trata os NaNs gerados na primeira linha pelo uso do shift(1)
df_features_raw = df_features_raw.bfill()

# 3. Construção dos arrays 3D
X_seq, y_seq, indices_validos = [], [], []

for tempo_alvo, row in df_alvos.iterrows():
    fim_idx = df_features_raw.index.get_loc(tempo_alvo)
    inicio_idx = fim_idx - seq_length + 1
    
    if inicio_idx >= 0:
        janela_3d = df_features_raw.iloc[inicio_idx : fim_idx + 1].values
        X_seq.append(janela_3d)
        y_seq.append(row[target_limpo])
        indices_validos.append(tempo_alvo)

X_seq = np.array(X_seq)
y_seq = np.array(y_seq).reshape(-1, 1)

# 4. Divisão Temporal (80% Treino / 20% Teste)
split_seq = int(len(X_seq) * 0.8)

X_train_seq, X_test_seq = X_seq[:split_seq], X_seq[split_seq:]
y_train_seq, y_test_seq = y_seq[:split_seq], y_seq[split_seq:]
idx_train, idx_test = indices_validos[:split_seq], indices_validos[split_seq:]

print(f"Shape X Treino: {X_train_seq.shape} | Shape y Treino: {y_train_seq.shape}")
print(f"Shape X Teste:  {X_test_seq.shape} | Shape y Teste:  {y_test_seq.shape}")

### DataLoaders

In [ ]:
# Criação dos DataLoaders do PyTorch
import torch
from torch.utils.data import TensorDataset, DataLoader

batch_size = 128

X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_seq, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_seq, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

print("DataLoaders prontos.")

### Arquitetura shallow

In [ ]:
# Definição da arquitetura Deep Learning Ultraleve
import torch.nn as nn
import torch

class UltralightSoftSensorLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=8, num_layers=1):
        super(UltralightSoftSensorLSTM, self).__init__()
        
        # Mantém a Normalização por Instância (crucial para o concept drift)
        self.instance_norm = nn.InstanceNorm1d(input_dim, affine=False)
        
        # LSTM mínima: 8 neurônios. (O argumento dropout foi removido pois o PyTorch só aplica se num_layers > 1)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        
        # Saída direta com forte dropout antes da predição final
        self.dropout = nn.Dropout(0.4)
        self.fc_final = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape inicial: [batch, seq_len, features]
        
        # 1. Executa a Normalização por Instância
        x = x.permute(0, 2, 1)
        x = self.instance_norm(x)
        x = x.permute(0, 2, 1) 
        
        # 2. Passa pela estrutura recorrente da LSTM
        lstm_out, _ = self.lstm(x) 
        
        # 3. Pega estritamente o último passo no tempo (minuto 180) da sequência
        ultimo_instante = lstm_out[:, -1, :] # shape: [batch, hidden_dim]
        
        # 4. Camada de saída única
        out = self.dropout(ultimo_instante)
        out = self.fc_final(out)
        
        return out

# Instancia a nova rede ultraleve
input_dim = X_train_tensor.shape[2]
modelo_ultraleve = UltralightSoftSensorLSTM(input_dim)
print(modelo_ultraleve)

### Treinamento

In [ ]:
# Loop de treinamento com tratamento de estabilidade
import torch.optim as optim

optimizer = optim.AdamW(modelo_ultraleve.parameters(), lr=0.0005, weight_decay=0.01)
criterion = nn.MSELoss()

epochs = 500
train_losses, test_losses = [], []

print("Treinando LSTM com InstanceNorm e Atenção...")

for epoch in range(epochs):
    modelo_ultraleve.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = modelo_ultraleve(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        # Clipping para evitar explosão de gradiente com a inserção da atenção
        torch.nn.utils.clip_grad_norm_(modelo_ultraleve.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    
    modelo_ultraleve.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = modelo_ultraleve(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    epoch_val_loss = val_loss / len(test_loader.dataset)
    test_losses.append(epoch_val_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Época {epoch+1:03d}/{epochs} | Loss Treino: {epoch_loss:.4f} | Loss Teste: {epoch_val_loss:.4f}")

### Avaliação

In [ ]:
# Previsões da LSTM avançada e avaliação
modelo_ultraleve.eval()
with torch.no_grad():
    preds_treino = modelo_ultraleve(X_train_tensor).numpy().flatten()
    preds_teste = modelo_ultraleve(X_test_tensor).numpy().flatten()

y_train_series = pd.Series(y_train_seq.flatten(), index=idx_train)
y_test_series = pd.Series(y_test_seq.flatten(), index=idx_test)

avaliar_modelo("LSTM Shallow", y_train_series, preds_treino, y_test_series, preds_teste)
plotar_resultados(y_train_series, preds_treino, y_test_series, preds_teste, "LSTM Shallow")

## LSTM com Target Differencing

A rede prevê a *variação* em relação ao último laudo; o valor absoluto é reconstruído depois. Ajuda quando o nível absoluto sofre drift.

### Sequências 3D (target diferenciado)

In [ ]:
# Transformação com Target Differencing
import numpy as np
import pandas as pd

print("Construindo sequências 3D para prever a VARIAÇÃO do processo...")

seq_length = 180

# 1. Prepara os alvos calculando a diferença para o laudo anterior
df_alvos = df_dataset[[target_limpo]].dropna().copy()
df_alvos['valor_anterior'] = df_alvos[target_limpo].shift(1)
df_alvos['delta_target'] = df_alvos[target_limpo].diff()

# Remove a primeira linha pois não tem laudo anterior para calcular o delta
df_alvos = df_alvos.dropna()

df_features_raw = df_dataset[features_processo].ffill().bfill()

X_seq, y_delta_seq, y_prev_seq, indices_validos = [], [], [], []

for tempo_alvo, row in df_alvos.iterrows():
    fim_idx = df_features_raw.index.get_loc(tempo_alvo)
    inicio_idx = fim_idx - seq_length + 1
    
    if inicio_idx >= 0:
        janela_3d = df_features_raw.iloc[inicio_idx : fim_idx + 1].values
        X_seq.append(janela_3d)
        y_delta_seq.append(row['delta_target'])      # A rede vai prever ISSO
        y_prev_seq.append(row['valor_anterior'])     # Guardamos para reconstruir o valor real depois
        indices_validos.append(tempo_alvo)

X_seq = np.array(X_seq)
y_delta_seq = np.array(y_delta_seq).reshape(-1, 1)
y_prev_seq = np.array(y_prev_seq).reshape(-1, 1)

split_seq = int(len(X_seq) * 0.8)

X_train_seq, X_test_seq = X_seq[:split_seq], X_seq[split_seq:]
y_train_delta, y_test_delta = y_delta_seq[:split_seq], y_delta_seq[split_seq:]

# Guardamos os valores anteriores de teste para avaliação
y_prev_train, y_prev_test = y_prev_seq[:split_seq], y_prev_seq[split_seq:]
idx_train, idx_test = indices_validos[:split_seq], indices_validos[split_seq:]

print(f"Shape X Treino: {X_train_seq.shape} | Shape y (Delta) Treino: {y_train_delta.shape}")

### DataLoaders

In [ ]:
# DataLoaders com o Target transformado
import torch
from torch.utils.data import TensorDataset, DataLoader

batch_size = 32

X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_delta_tensor = torch.tensor(y_train_delta, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_delta_tensor = torch.tensor(y_test_delta, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_delta_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_delta_tensor), batch_size=batch_size, shuffle=False)

print("DataLoaders prontos para treinar as variações.")

### Arquitetura

In [ ]:
# Definição da nova arquitetura da rede neural
import torch.nn as nn
import torch

class AdvancedSoftSensorLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super(AdvancedSoftSensorLSTM, self).__init__()
        
        # Solução 1: Instance Normalization (Normaliza cada sensor dentro da sua própria janela)
        # affine=False garante que não adicionamos parâmetros lineares fixos que possam enviesar regimes diferentes
        self.instance_norm = nn.InstanceNorm1d(input_dim, affine=False)
        
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        
        # Solução 3: Mecanismo de Atenção (Gera um score de importância para cada um dos 180 passos temporais)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        # x shape inicial: [batch, seq_len, features]
        
        # 1. Executa a Normalização por Instância
        # O módulo exige a dimensão de features na segunda posição: [batch, features, seq_len]
        x = x.permute(0, 2, 1)
        x = self.instance_norm(x)
        x = x.permute(0, 2, 1) # Retorna para: [batch, seq_len, features]
        
        # 2. Passa pela estrutura recorrente da LSTM
        lstm_out, _ = self.lstm(x) # lstm_out shape: [batch, seq_len, hidden_dim]
        
        # 3. Aplica o cálculo do Mecanismo de Atenção
        attn_scores = self.attention(lstm_out) # shape: [batch, seq_len, 1]
        attn_weights = torch.softmax(attn_scores, dim=1) # Normaliza os pesos no tempo
        
        # Multiplica os pesos pelos estados ocultos correspondentes e soma todo o histórico (Context Vector)
        context = torch.sum(attn_weights * lstm_out, dim=1) # shape: [batch, hidden_dim]
        
        # 4. Camadas de saída densas
        out = self.fc1(context)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

input_dim = X_train_tensor.shape[2]
modelo_avancado = AdvancedSoftSensorLSTM(input_dim)
print(modelo_avancado)

### Treinamento

In [ ]:
# Loop de treinamento com tratamento de estabilidade
import torch.optim as optim

optimizer = optim.AdamW(modelo_avancado.parameters(), lr=0.0005, weight_decay=0.01)
criterion = nn.MSELoss()

epochs = 100
train_losses, test_losses = [], []

print("Treinando LSTM com InstanceNorm e Atenção...")

for epoch in range(epochs):
    modelo_avancado.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = modelo_avancado(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        # Clipping para evitar explosão de gradiente com a inserção da atenção
        torch.nn.utils.clip_grad_norm_(modelo_avancado.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    
    modelo_avancado.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = modelo_avancado(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    epoch_val_loss = val_loss / len(test_loader.dataset)
    test_losses.append(epoch_val_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Época {epoch+1:03d}/{epochs} | Loss Treino: {epoch_loss:.4f} | Loss Teste: {epoch_val_loss:.4f}")

### Avaliação

In [ ]:
# Reconstrói o valor absoluto (valor anterior + variação prevista) e avalia
modelo_avancado.eval()
with torch.no_grad():
    preds_delta_treino = modelo_avancado(X_train_tensor).numpy().flatten()
    preds_delta_teste = modelo_avancado(X_test_tensor).numpy().flatten()

preds_absolutas_treino = y_prev_train.flatten() + preds_delta_treino
preds_absolutas_teste = y_prev_test.flatten() + preds_delta_teste

y_real_treino = y_prev_train.flatten() + y_train_delta.flatten()
y_real_teste = y_prev_test.flatten() + y_test_delta.flatten()

y_train_series = pd.Series(y_real_treino, index=idx_train)
y_test_series = pd.Series(y_real_teste, index=idx_test)

avaliar_modelo("LSTM Differencing", y_train_series, preds_absolutas_treino, y_test_series, preds_absolutas_teste)
plotar_resultados(y_train_series, preds_absolutas_treino, y_test_series, preds_absolutas_teste, "LSTM Differencing")

## LSTM Restrita (anti-overfitting)

Versão de capacidade reduzida e regularização forte para conter o overfitting observado na LSTM avançada. Seção autossuficiente: reconstrói as próprias sequências 3D com *target differencing* e os DataLoaders, dependendo apenas da base `df_dataset` e das funções utilitárias.

### Sequências 3D (target diferenciado)

In [ ]:
# Reconstrói as sequências 3D com Target Differencing para esta seção rodar de forma independente
import numpy as np
import pandas as pd

print("Construindo sequências 3D (LSTM Restrita)...")

seq_length = 180

# Variáveis de processo (exclui o target original e o target limpo)
features_processo = [col for col in df_dataset.columns if col not in [target_col, target_limpo]]

# Alvos: variação (delta) em relação ao laudo anterior, guardando o valor anterior para reconstrução
df_alvos = df_dataset[[target_limpo]].dropna().copy()
df_alvos['valor_anterior'] = df_alvos[target_limpo].shift(1)
df_alvos['delta_target'] = df_alvos[target_limpo].diff()
df_alvos = df_alvos.dropna()

df_features_raw = df_dataset[features_processo].ffill().bfill()

X_seq, y_delta_seq, y_prev_seq, indices_validos = [], [], [], []
for tempo_alvo, row in df_alvos.iterrows():
    fim_idx = df_features_raw.index.get_loc(tempo_alvo)
    inicio_idx = fim_idx - seq_length + 1
    if inicio_idx >= 0:
        X_seq.append(df_features_raw.iloc[inicio_idx:fim_idx + 1].values)
        y_delta_seq.append(row['delta_target'])
        y_prev_seq.append(row['valor_anterior'])
        indices_validos.append(tempo_alvo)

X_seq = np.array(X_seq)
y_delta_seq = np.array(y_delta_seq).reshape(-1, 1)
y_prev_seq = np.array(y_prev_seq).reshape(-1, 1)

# Split temporal 80/20
split_seq = int(len(X_seq) * 0.8)
X_train_seq, X_test_seq = X_seq[:split_seq], X_seq[split_seq:]
y_train_delta, y_test_delta = y_delta_seq[:split_seq], y_delta_seq[split_seq:]
y_prev_train, y_prev_test = y_prev_seq[:split_seq], y_prev_seq[split_seq:]
idx_train, idx_test = indices_validos[:split_seq], indices_validos[split_seq:]

print(f"Shape X Treino: {X_train_seq.shape} | Shape y (Delta) Treino: {y_train_delta.shape}")

### DataLoaders

In [ ]:
# DataLoaders próprios da seção (target diferenciado)
import torch
from torch.utils.data import TensorDataset, DataLoader

batch_size = 32

X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_delta_tensor = torch.tensor(y_train_delta, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_delta_tensor = torch.tensor(y_test_delta, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_delta_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_delta_tensor), batch_size=batch_size, shuffle=False)

print("DataLoaders prontos para a LSTM Restrita.")

### Arquitetura

In [ ]:
# Arquitetura LSTM Fortemente Restrita (Anti-Overfitting)
import torch.nn as nn
import torch

class RestrictedSoftSensorLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=8, num_layers=1):
        super(RestrictedSoftSensorLSTM, self).__init__()
        
        # Mantém a Normalização por Instância (crucial para o concept drift)
        self.instance_norm = nn.InstanceNorm1d(input_dim, affine=False)
        
        # LSTM mínima: 8 neurônios. (O argumento dropout foi removido pois o PyTorch só aplica se num_layers > 1)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        
        # Saída direta com forte dropout antes da predição final
        self.dropout = nn.Dropout(0.4)
        self.fc_final = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape inicial: [batch, seq_len, features]
        
        # 1. Executa a Normalização por Instância
        x = x.permute(0, 2, 1)
        x = self.instance_norm(x)
        x = x.permute(0, 2, 1) 
        
        # 2. Passa pela estrutura recorrente da LSTM
        lstm_out, _ = self.lstm(x) 
        
        # 3. Pega estritamente o último passo no tempo (minuto 180) da sequência
        ultimo_instante = lstm_out[:, -1, :] # shape: [batch, hidden_dim]
        
        # 4. Camada de saída única
        out = self.dropout(ultimo_instante)
        out = self.fc_final(out)
        
        return out

input_dim = X_train_tensor.shape[2]
modelo_restrito = RestrictedSoftSensorLSTM(input_dim)
print(modelo_restrito)

### Treinamento

In [ ]:
# Treinamento Otimizado com Alta Regularização
import torch.optim as optim

# Weight_decay (L2) elevado a 0.1 para punir pesos grandes e forçar suavização
optimizer = optim.AdamW(modelo_restrito.parameters(), lr=0.0005, weight_decay=0.1)
criterion = nn.MSELoss()

epochs = 100
train_losses, test_losses = [], []

print("Treinando LSTM Restrita...")

for epoch in range(epochs):
    modelo_restrito.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = modelo_restrito(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(modelo_restrito.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    
    modelo_restrito.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = modelo_restrito(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    epoch_val_loss = val_loss / len(test_loader.dataset)
    test_losses.append(epoch_val_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Época {epoch+1:03d}/{epochs} | Loss Treino: {epoch_loss:.4f} | Loss Teste: {epoch_val_loss:.4f}")

### Avaliação

In [ ]:
# Reconstrói o valor absoluto e avalia a LSTM restrita
modelo_restrito.eval()
with torch.no_grad():
    preds_delta_treino = modelo_restrito(X_train_tensor).numpy().flatten()
    preds_delta_teste = modelo_restrito(X_test_tensor).numpy().flatten()

preds_absolutas_treino = y_prev_train.flatten() + preds_delta_treino
preds_absolutas_teste = y_prev_test.flatten() + preds_delta_teste

y_real_treino = y_prev_train.flatten() + y_train_delta.flatten()
y_real_teste = y_prev_test.flatten() + y_test_delta.flatten()

y_train_series = pd.Series(y_real_treino, index=idx_train)
y_test_series = pd.Series(y_real_teste, index=idx_test)

avaliar_modelo("LSTM Restrita", y_train_series, preds_absolutas_treino, y_test_series, preds_absolutas_teste)
plotar_resultados(y_train_series, preds_absolutas_treino, y_test_series, preds_absolutas_teste, "LSTM Restrita")

# Experimento 4 - Series temporais: ARIMA e ARIMAX

O CaO Livre e medido em laboratorio de forma **irregular** (intervalo mediano de dezenas de
minutos). Modelos classicos como ARIMA pressupoem amostragem regular, entao aqui tratamos a
**sequencia de medicoes** como a serie temporal (indexada pela ordem da coleta, nao pelo relogio).
Essa e a forma natural de capturar a dinamica do laudo:

- **ARIMA(p,d,q)** modela o CaO Livre a partir dos seus proprios valores passados. Um `ARIMA(1,0,0)`
  com constante e exatamente a *regressao a media do ultimo laudo* (`y(t-1)`), que nas fases
  anteriores se mostrou o sinal mais robusto.
- **ARIMAX** acrescenta variaveis **exogenas** de processo (medias moveis das tags do forno),
  testando se elas agregam informacao **incremental** sobre a estrutura autoregressiva.

Avaliamos com previsao **1-passo-a-frente** (a cada novo laudo, prevemos o proximo) e, ao final,
com **walk-forward** (varias janelas) para um R2 mais honesto que o holdout unico.

In [ ]:
# Preparacao da serie de laudos + baselines honestos
# (statsmodels ja consta no requirements; se faltar: pip install statsmodels)
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    from statsmodels.tsa.statespace.sarimax import SARIMAX
import numpy as np, pandas as pd

# Serie indexada pela ORDEM da coleta (uma observacao por laudo real de laboratorio)
serie_lab = df_dataset[target_limpo].dropna()
instantes_lab = serie_lab.index                  # carimbos de tempo reais das coletas
y_serie = serie_lab.reset_index(drop=True)        # serie 0..N-1 para os modelos ARIMA
N = len(y_serie)

# Reaproveita a data de corte definida no calculo de Informacao Mutua (sem vazamento).
# Fallback: 80% das coletas, caso a celula da MI nao tenha sido executada.
try:
    k = int((instantes_lab <= train_cutoff_date).sum())
except NameError:
    k = int(N * 0.8)
k = max(2, min(k, N - 1))

y_tr, y_te = y_serie.iloc[:k], y_serie.iloc[k:]
idx_tr, idx_te = instantes_lab[:k], instantes_lab[k:]
print(f"Coletas totais: {N} | treino: {k} | teste: {N-k}")
print(f"Autocorrelacao lag-1 do laudo: {y_serie.autocorr(1):.3f} "
      f"(quanto maior, mais previsivel pelo passado)")

# --- Baselines de referencia ---
# Persistencia: prever o ultimo laudo conhecido y(t-1)
pers_tr = y_tr.shift(1)
pers_te = y_serie.shift(1).iloc[k:]
m_tr = ~pers_tr.isna()
avaliar_modelo("Baseline Persistencia y(t-1)",
               y_tr[m_tr], pers_tr[m_tr], y_te, pers_te)

# Media do treino (modelo "burro" de referencia para o R2)
media_tr = y_tr.mean()
avaliar_modelo("Baseline Media do treino",
               y_tr, np.full(len(y_tr), media_tr),
               y_te, np.full(len(y_te), media_tr))

In [ ]:
# ARIMA (sem exogenas): selecao de ordem por AIC no TREINO + previsao 1-passo no teste
import numpy as np, pandas as pd

ordens_candidatas = [(1,0,0), (2,0,0), (1,0,1), (2,0,1), (1,1,1), (2,1,1)]

melhor = None
linhas_aic = []
for ordem in ordens_candidatas:
    trend = 'c' if ordem[1] == 0 else 'n'   # constante so faz sentido sem diferenciacao
    try:
        fit = SARIMAX(y_tr, order=ordem, trend=trend,
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        linhas_aic.append({'ordem': str(ordem), 'AIC': fit.aic, 'BIC': fit.bic})
        if melhor is None or fit.aic < melhor[1]:
            melhor = (ordem, fit.aic, trend)
    except Exception as e:
        linhas_aic.append({'ordem': str(ordem), 'AIC': np.nan, 'BIC': str(e)[:40]})

display(pd.DataFrame(linhas_aic).sort_values('AIC').reset_index(drop=True))
ordem_arima, _, trend_arima = melhor
print(f"Melhor ordem por AIC: ARIMA{ordem_arima} (trend='{trend_arima}')")

# Reestima os parametros no treino e os aplica a serie completa para gerar
# previsoes 1-passo-a-frente no teste (dynamic=False usa o y real ate t-1 -> sem vazamento)
fit_tr = SARIMAX(y_tr, order=ordem_arima, trend=trend_arima,
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
filtro = SARIMAX(y_serie, order=ordem_arima, trend=trend_arima,
                 enforce_stationarity=False, enforce_invertibility=False).filter(fit_tr.params)

pred_in = filtro.get_prediction(start=0, end=k-1, dynamic=False).predicted_mean
pred_out = filtro.get_prediction(start=k, end=N-1, dynamic=False).predicted_mean

# alinha indices temporais reais para o plot/registro
yt_tr = pd.Series(y_tr.values, index=idx_tr); pt_tr = pd.Series(pred_in.values, index=idx_tr)
yt_te = pd.Series(y_te.values, index=idx_te); pt_te = pd.Series(pred_out.values, index=idx_te)
# descarta a 1a amostra do treino (sem passado para prever)
avaliar_modelo(f"ARIMA{ordem_arima}", yt_tr.iloc[1:], pt_tr.iloc[1:], yt_te, pt_te)
plotar_resultados(yt_tr.iloc[1:], pt_tr.iloc[1:].values, yt_te, pt_te.values, f"ARIMA{ordem_arima}")

In [ ]:
# ARIMAX: ARIMA + variaveis exogenas de processo (medias moveis de 180 min)
import numpy as np, pandas as pd

# 1. Exogenas candidatas: media movel de 180 min de cada tag de processo,
#    amostrada nos instantes de laudo. Capta a "condicao media" do forno antes da coleta.
feats_proc = [c for c in df_dataset.columns if c not in [target_col, target_limpo]]
roll = df_dataset[feats_proc].rolling(window=180, min_periods=30).mean()
exog_full = roll.loc[instantes_lab].reset_index(drop=True).ffill().bfill()

# 2. Selecao das exogenas por |correlacao| com o alvo - calculada SOMENTE no treino
corr_tr = exog_full.iloc[:k].corrwith(y_tr).abs().sort_values(ascending=False)
top_exog = corr_tr.head(8).index.tolist()
print("Exogenas selecionadas (corr no treino):")
display(corr_tr.head(8).round(3).to_frame("|corr| treino"))

# 3. Padronizacao ajustada SO no treino (sem vazamento)
exog = exog_full[top_exog]
mu, sd = exog.iloc[:k].mean(), exog.iloc[:k].std().replace(0, 1.0)
exog_z = (exog - mu) / sd
ex_tr, ex_te = exog_z.iloc[:k], exog_z.iloc[k:]

# 4. Ajuste no treino e previsao 1-passo no teste (mesma ordem AR escolhida acima)
fit_x = SARIMAX(y_tr, exog=ex_tr, order=ordem_arima, trend=trend_arima,
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=300)
filtro_x = SARIMAX(y_serie, exog=exog_z, order=ordem_arima, trend=trend_arima,
                   enforce_stationarity=False, enforce_invertibility=False).filter(fit_x.params)

px_in = filtro_x.get_prediction(start=0, end=k-1, exog=ex_tr, dynamic=False).predicted_mean
px_out = filtro_x.get_prediction(start=k, end=N-1, dynamic=False).predicted_mean

yt_tr = pd.Series(y_tr.values, index=idx_tr); ptx_tr = pd.Series(px_in.values, index=idx_tr)
yt_te = pd.Series(y_te.values, index=idx_te); ptx_te = pd.Series(px_out.values, index=idx_te)
avaliar_modelo(f"ARIMAX{ordem_arima} (+processo)", yt_tr.iloc[1:], ptx_tr.iloc[1:], yt_te, ptx_te)
plotar_resultados(yt_tr.iloc[1:], ptx_tr.iloc[1:].values, yt_te, ptx_te.values, f"ARIMAX{ordem_arima}")

## Validacao walk-forward (R2 mais honesto)

O holdout unico (ultimos 20%) e sensivel ao regime daquela janela especifica. O **walk-forward**
reestima o modelo em janela crescente e mede o erro em varios blocos consecutivos, dando uma media
mais representativa. Comparamos ARIMA contra a persistencia em 5 blocos.

In [ ]:
# Walk-forward expanding (5 blocos) - diagnostico, nao registrado no ranking final
import numpy as np, pandas as pd

def _r2(yt, yp):
    yt, yp = np.asarray(yt, float), np.asarray(yp, float)
    return 1 - np.sum((yt-yp)**2) / np.sum((yt-yt.mean())**2)
def _rmse(yt, yp):
    return float(np.sqrt(np.mean((np.asarray(yt,float)-np.asarray(yp,float))**2)))

n_folds = 5
inicio = int(N * 0.5)
passo = (N - inicio) // n_folds
linhas = []
for f in range(n_folds):
    a = inicio + f*passo
    b = (inicio + (f+1)*passo) if f < n_folds-1 else N
    ytr_f, yte_f = y_serie.iloc[:a], y_serie.iloc[a:b]
    fit_f = SARIMAX(ytr_f, order=ordem_arima, trend=trend_arima,
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    flt = SARIMAX(y_serie, order=ordem_arima, trend=trend_arima,
                  enforce_stationarity=False, enforce_invertibility=False).filter(fit_f.params)
    fc = flt.get_prediction(start=a, end=b-1, dynamic=False).predicted_mean.values
    pers = y_serie.shift(1).iloc[a:b].values
    linhas.append({'fold': f, 'n_teste': len(yte_f),
                   f'R2 ARIMA{ordem_arima}': _r2(yte_f, fc), 'RMSE ARIMA': _rmse(yte_f, fc),
                   'R2 Persist.': _r2(yte_f, pers), 'RMSE Persist.': _rmse(yte_f, pers)})

df_wf = pd.DataFrame(linhas)
display(df_wf.style.format({c: "{:.3f}" for c in df_wf.columns if c not in ['fold','n_teste']}))
print(f"MEDIA  ->  ARIMA R2={df_wf[f'R2 ARIMA{ordem_arima}'].mean():.3f} "
      f"RMSE={df_wf['RMSE ARIMA'].mean():.3f}  |  "
      f"Persistencia R2={df_wf['R2 Persist.'].mean():.3f} RMSE={df_wf['RMSE Persist.'].mean():.3f}")

# Experimento 5 - Modelo otimizado (regularizacao + multi-janela + ensemble)

Os experimentos anteriores mostraram dois problemas: (1) arvores/LSTM **decoram** o treino
(R2 treino ~0,9, teste ~0,1) por excesso de features; (2) o **holdout unico** e instavel,
pois a ultima janela e um regime mais dificil. Esta etapa ataca os dois pontos para ganhar R2
de forma **honesta**:

1. **Backbone autoregressivo**: `y_lag1` e `y_lag2` (ultimos laudos) - o sinal mais forte.
2. **Features de processo multi-janela**: media (60/180/360 min), desvio (180 min), valor
   instantaneo e *slope* (tendencia) de cada tag, amostradas no instante do laudo.
3. **Selecao honesta**: as `k` melhores features por correlacao calculada **so no treino**.
4. **Regularizacao forte + early stopping** (XGB raso, ElasticNet) para nao decorar.
5. **Ensemble** (media de XGB + ElasticNet): combina o nao-linear e o linear, reduzindo variancia.

Avaliamos no **holdout** (para a tabela de ranking) e, principalmente, em **walk-forward**
(5 janelas), que e a metrica confiavel para decidir o melhor modelo.

In [ ]:
# --- Engenharia de features: backbone AR + estatisticas multi-janela de processo ---
import numpy as np, pandas as pd

# Serie de laudos reais (de-plato) e seus instantes
serie_lab_opt = df_dataset[target_limpo].dropna()
inst_opt = serie_lab_opt.index
y_opt = serie_lab_opt.reset_index(drop=True)
N_opt = len(y_opt)

feats_proc = [c for c in df_dataset.columns if c not in [target_col, target_limpo]]

# Estatisticas em multiplas janelas (capturam a "condicao media" e a variabilidade do forno)
mean60  = df_dataset[feats_proc].rolling(60,  min_periods=10).mean()
mean180 = df_dataset[feats_proc].rolling(180, min_periods=30).mean()
mean360 = df_dataset[feats_proc].rolling(360, min_periods=60).mean()
std180  = df_dataset[feats_proc].rolling(180, min_periods=30).std()
# slope: tendencia recente (media dos ultimos 60 min menos a de 120 min atras)
slope60 = mean60 - mean60.shift(120)

X_opt = pd.concat([
    mean60.add_suffix('_m60'), mean180.add_suffix('_m180'), mean360.add_suffix('_m360'),
    std180.add_suffix('_s180'), df_dataset[feats_proc].add_suffix('_now'),
    slope60.add_suffix('_slope'),
], axis=1).loc[inst_opt].reset_index(drop=True)
X_opt = X_opt.ffill().bfill().fillna(0)

# Backbone autoregressivo
ylag1_opt = y_opt.shift(1)
ylag2_opt = y_opt.shift(2)

print(f"Laudos: {N_opt} | features de processo geradas: {X_opt.shape[1]}")

# Corte treino/teste coerente com o resto do notebook (mesma data, sem vazamento)
try:
    k_split = int((inst_opt <= train_cutoff_date).sum())
except NameError:
    k_split = int(N_opt * 0.8)
k_split = max(3, min(k_split, N_opt - 1))
idx_tr_opt, idx_te_opt = inst_opt[:k_split], inst_opt[k_split:]
print(f"Treino: {k_split} | Teste: {N_opt - k_split}")

In [ ]:
# --- Selecao honesta (so no treino) e montagem das matrizes ---
from sklearn.preprocessing import StandardScaler

K_FEATURES = 50  # nº de features de processo mantidas (selecionadas por correlacao no treino)

def selecionar_features(ate_idx, k=K_FEATURES):
    """Top-k features de processo por |correlacao| com o alvo, usando SO o periodo de treino."""
    corr = X_opt.iloc[1:ate_idx].corrwith(y_opt.iloc[1:ate_idx]).abs()
    return corr.sort_values(ascending=False).head(k).index.tolist()

cols_sel = selecionar_features(k_split)
print(f"Selecionadas {len(cols_sel)} features. Top 10:")
display(X_opt.iloc[1:k_split].corrwith(y_opt.iloc[1:k_split]).abs()
        .sort_values(ascending=False).head(10).round(3).to_frame('|corr| treino'))

# Matriz para modelos NAO-lineares (XGB): backbone + processo selecionado
Xfull = pd.concat([ylag1_opt.rename('y_lag1'), ylag2_opt.rename('y_lag2'),
                   X_opt[cols_sel]], axis=1).bfill()
# Matriz para o LINEAR (ElasticNet): mesma base, padronizada
Xlin = pd.concat([ylag1_opt.rename('y_lag1'), X_opt[cols_sel]], axis=1).bfill()

Xfull_tr, Xfull_te = Xfull.iloc[2:k_split], Xfull.iloc[k_split:]
Xlin_tr,  Xlin_te  = Xlin.iloc[2:k_split],  Xlin.iloc[k_split:]
y_tr_opt = y_opt.iloc[2:k_split]
y_te_opt = y_opt.iloc[k_split:]
yt_tr = pd.Series(y_tr_opt.values, index=idx_tr_opt[2:])
yt_te = pd.Series(y_te_opt.values, index=idx_te_opt)

scaler_lin = StandardScaler().fit(Xlin_tr)
Xlin_tr_z = scaler_lin.transform(Xlin_tr)
Xlin_te_z = scaler_lin.transform(Xlin_te)

In [ ]:
# --- Modelo 1: ElasticNet (linear regularizado, interpretavel) ---
from sklearn.linear_model import ElasticNetCV

enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=4, max_iter=5000, random_state=0)
enet.fit(Xlin_tr_z, y_tr_opt)
pred_enet_tr = enet.predict(Xlin_tr_z)
pred_enet_te = enet.predict(Xlin_te_z)

avaliar_modelo("Exp5 ElasticNet (AR + processo)", yt_tr, pred_enet_tr, yt_te, pred_enet_te)
plotar_resultados(yt_tr, pred_enet_tr, yt_te, pred_enet_te, "Exp5 ElasticNet")

In [ ]:
# --- Modelo 2: XGBoost FORTEMENTE regularizado (raso + subsample + L1/L2) ---
import xgboost as xgb

xgb_reg = xgb.XGBRegressor(
    max_depth=3, n_estimators=300, learning_rate=0.03,
    subsample=0.7, colsample_bytree=0.6,
    reg_lambda=3.0, reg_alpha=0.5, min_child_weight=10,
    random_state=0, n_jobs=-1,
)
xgb_reg.fit(Xfull_tr, y_tr_opt)
pred_xgb_tr = xgb_reg.predict(Xfull_tr)
pred_xgb_te = xgb_reg.predict(Xfull_te)

avaliar_modelo("Exp5 XGBoost regularizado", yt_tr, pred_xgb_tr, yt_te, pred_xgb_te)
plotar_resultados(yt_tr, pred_xgb_tr, yt_te, pred_xgb_te, "Exp5 XGBoost regularizado")

# Importancia das variaveis (quais janelas/tags o modelo mais usa)
display(tabela_importancias(xgb_reg, Xfull.columns, top=15))

In [ ]:
# --- Modelo 3: ENSEMBLE (media de XGB + ElasticNet) ---
# Combina o nao-linear (XGB) com o linear (ElasticNet): reduz variancia e e o mais
# estavel em walk-forward.
pred_ens_tr = 0.5 * pred_xgb_tr + 0.5 * pred_enet_tr
pred_ens_te = 0.5 * pred_xgb_te + 0.5 * pred_enet_te

avaliar_modelo("Exp5 ENSEMBLE (XGB + ElasticNet)", yt_tr, pred_ens_tr, yt_te, pred_ens_te)
plotar_resultados(yt_tr, pred_ens_tr, yt_te, pred_ens_te, "Exp5 ENSEMBLE")

## Validacao walk-forward (metrica de decisao)

O holdout unico cai num regime dificil e penaliza o XGB; a media de 5 janelas walk-forward
mostra o desempenho real. **E por esta tabela que se deve escolher o modelo de producao.**

In [ ]:
# Walk-forward (5 blocos) do Exp5 vs AR(1) - re-treina tudo em cada janela (sem vazamento)
import numpy as np, pandas as pd
from sklearn.linear_model import ElasticNetCV, RidgeCV
import xgboost as xgb

def _r2(yt, yp):
    yt, yp = np.asarray(yt, float), np.asarray(yp, float)
    return 1 - np.sum((yt-yp)**2) / np.sum((yt-yt.mean())**2)
def _rmse(yt, yp):
    return float(np.sqrt(np.mean((np.asarray(yt,float)-np.asarray(yp,float))**2)))

def _fit_pred(a, b):
    """Treina ENet, XGB e o ensemble com dados ate 'a' e preve [a:b]."""
    cols = selecionar_features(a)
    Xf = pd.concat([ylag1_opt.rename('y_lag1'), ylag2_opt.rename('y_lag2'), X_opt[cols]], axis=1).bfill()
    Xl = pd.concat([ylag1_opt.rename('y_lag1'), X_opt[cols]], axis=1).bfill()
    sc = StandardScaler().fit(Xl.iloc[1:a])
    en = ElasticNetCV(l1_ratio=[0.1,0.5,0.9], cv=4, max_iter=5000, random_state=0).fit(sc.transform(Xl.iloc[1:a]), y_opt.iloc[1:a])
    xg = xgb.XGBRegressor(max_depth=3, n_estimators=300, learning_rate=0.03, subsample=0.7,
                          colsample_bytree=0.6, reg_lambda=3.0, reg_alpha=0.5, min_child_weight=10,
                          random_state=0, n_jobs=-1).fit(Xf.iloc[2:a], y_opt.iloc[2:a])
    rg = RidgeCV(alphas=[0.1,1,10]).fit(ylag1_opt.iloc[1:a].values.reshape(-1,1), y_opt.iloc[1:a].values)
    p_en = en.predict(sc.transform(Xl.iloc[a:b]))
    p_xg = xg.predict(Xf.iloc[a:b])
    p_ar = rg.predict(ylag1_opt.iloc[a:b].values.reshape(-1,1))
    return p_ar, p_en, p_xg, 0.5*p_en + 0.5*p_xg

n_folds, inicio = 5, int(N_opt * 0.5)
passo = (N_opt - inicio) // n_folds
linhas = []
for f in range(n_folds):
    a = inicio + f*passo
    b = inicio + (f+1)*passo if f < n_folds-1 else N_opt
    yte = y_opt.iloc[a:b]
    p_ar, p_en, p_xg, p_ens = _fit_pred(a, b)
    linhas.append({'fold': f, 'n_teste': len(yte),
                   'R2 AR(1)': _r2(yte, p_ar), 'R2 ENet': _r2(yte, p_en),
                   'R2 XGB': _r2(yte, p_xg), 'R2 ENSEMBLE': _r2(yte, p_ens),
                   'RMSE ENSEMBLE': _rmse(yte, p_ens)})

df_wf5 = pd.DataFrame(linhas)
display(df_wf5.style.format({c: "{:.3f}" for c in df_wf5.columns if c not in ['fold','n_teste']}))
print("MEDIA walk-forward R2  ->  "
      f"AR(1)={df_wf5['R2 AR(1)'].mean():.3f} | ENet={df_wf5['R2 ENet'].mean():.3f} | "
      f"XGB={df_wf5['R2 XGB'].mean():.3f} | ENSEMBLE={df_wf5['R2 ENSEMBLE'].mean():.3f}")

# Comparativo final dos experimentos

Ranking de todos os modelos avaliados, ordenado pelo menor RMSE de teste.

In [ ]:
# Ranking de todos os experimentos avaliados (menor RMSE de teste primeiro)
df_comparativo = tabela_resultados()
display(
    df_comparativo.style
    .format({c: "{:.4f}" for c in df_comparativo.columns if c != 'Modelo'})
    .background_gradient(cmap='RdYlGn_r', subset=['RMSE Teste', 'MAE Teste'])
    .background_gradient(cmap='RdYlGn', subset=['R2 Teste'])
)